In [8]:
import os
import sys
from pathlib import Path

# Lock working directory to main project root (2 levels up from src/notebook/)
current_dir = Path.cwd()
if current_dir.name == "notebook" and current_dir.parent.name == "src":
    project_root = current_dir.parent.parent
    os.chdir(project_root)
elif current_dir.name == "src":
    project_root = current_dir.parent
    os.chdir(project_root)
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Kernel working directory locked to:", Path.cwd())

Kernel working directory locked to: /home/s2025ae05884/workspace/heart-disease-predict


In [9]:
# %%writefile src/download_data.py
# imports
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
def run_eda(data_path="data/processed/heart_disease_cleaned.csv", output_dir="screenshots"):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(data_path)

    # Set a clean, professional visual theme
    sns.set_theme(style="whitegrid", palette="muted")

    # 1. Class Balance Visualization
    plt.figure(figsize=(6, 4))
    ax = sns.countplot(data=df, x="target", hue="target", palette=["#2b5c8f", "#d95f02"], legend=False)
    plt.title("Target Class Distribution (0 = No Disease, 1 = Heart Disease)", fontsize=12, fontweight="bold")
    plt.xlabel("Heart Disease Status")
    plt.ylabel("Patient Count")
    for container in ax.containers:
        ax.bar_label(container)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "01_class_balance.png"), dpi=300)
    plt.close()

    # 2. Continuous Features Histograms & KDE
    continuous_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    for i, col in enumerate(continuous_cols):
        sns.histplot(data=df, x=col, hue="target", kde=True, ax=axes[i], palette="Set1", element="step")
        axes[i].set_title(f"Distribution of {col.upper()} by Target", fontweight="bold")
    # Hide the unused 6th subplot
    fig.delaxes(axes[5])
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "02_feature_histograms.png"), dpi=300)
    plt.close()

    # 3. Correlation Heatmap
    plt.figure(figsize=(12, 9))
    corr_matrix = df.corr()
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5, vmin=-1, vmax=1)
    plt.title("Feature Correlation Matrix (UCI Heart Disease)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "03_correlation_heatmap.png"), dpi=300)
    plt.close()

    # 4. Clinical Insight Plot (Age vs Max Heart Rate by Disease Status)
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="age", y="thalach", hue="target", style="sex", palette="Set1", s=70, alpha=0.85)
    plt.title("Clinical Interaction: Age vs. Max Heart Rate (thalach)", fontsize=12, fontweight="bold")
    plt.xlabel("Age (Years)")
    plt.ylabel("Maximum Heart Rate Achieved (bpm)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "04_clinical_interaction.png"), dpi=300)
    plt.close()

    print(f"EDA plots successfully generated and saved to '{output_dir}/'")


In [11]:
run_eda()

EDA plots successfully generated and saved to 'screenshots/'
